In [1]:
import pandas as pd
import zipfile
import os

# --- 1. List of your FAERS ZIP files ---
zip_files = [
    "faers_ascii_2024Q1.zip",
    "faers_ascii_2024Q2.zip",
    "faers_ascii_2024Q3.zip",
    "faers_ascii_2024Q4.zip",
    "faers_ascii_2025q1 (1).zip",
    "faers_ascii_2025q2 (2).zip",
    "faers_ascii_2025q3 (2).zip",
]

# --- 2. Columns to drop ---
unwanted_cols = [
    'val_vbm', 'dose_vbm', 'cum_dose_chr', 'cum_dose_unit',
    'dechal', 'rechal', 'lot_num', 'exp_dt', 'nda_num'
]

# --- 3. Drug names to keep ---
drug_list = [
    "Lamotrigine", "Levetiracetam", "Topiramate", "Gabapentin", "Pregabalin",
    "Oxcarbazepine", "Zonisamide", "Lacosamide", "Clobazam", "Phenytoin",
    "Carbamazepine", "Phenobarbital", "Valproic acid", "Sodium valproate",
    "Ethosuximide", "Levodopa + Carbidopa", "Bromocriptine", "Pramipexole",
    "Ropinirole", "Rotigotine", "Apomorphine", "Selegiline", "Rasagiline",
    "Safinamide", "Entacapone", "Tolcapone", "Trihexyphenidyl", "Benzhexol",
    "Benztropine", "Amantadine", "Donepezil", "Rivastigmine", "Galantamine",
    "Memantine", "Interferon beta-1a", "Interferon beta-1b", "Glatiramer acetate",
    "Fingolimod", "Teriflunomide", "Dimethyl fumarate", "Natalizumab",
    "Ocrelizumab", "Alemtuzumab", "Baclofen", "Tizanidine", "Modafinil",
    "Sumatriptan", "Rizatriptan", "Zolmitriptan", "Ibuprofen", "Naproxen",
    "Ergotamine", "Dihydroergotamine", "Metoclopramide", "Domperidone",
    "Propranolol", "Amitriptyline", "Candesartan", "Botulinum toxin A",
    "Nortriptyline", "Duloxetine", "Tetrabenazine", "Deutetrabenazine",
    "Haloperidol", "Risperidone", "Diazepam", "Pyridostigmine", "Neostigmine",
    "Prednisolone", "Azathioprine", "Mycophenolate mofetil", "Cyclosporine",
    "Eculizumab", "Rituximab", "Plasmapheresis", "IV immunoglobulin",
    "Zolpidem", "Zopiclone", "Melatonin", "Sodium oxybate", "Methylphenidate",
    "Ceftriaxone", "Vancomycin", "Acyclovir", "Amphotericin B", "Citicoline",
    "Piracetam", "Cerebrolysin", "Edaravone"
]



In [2]:
# Lowercase version for case-insensitive comparison
drug_list_lower = [d.strip().lower() for d in drug_list]

# --- Master DataFrame ---
master_drug_df = pd.DataFrame()   # <- initialize here

# --- Loop through ZIP files ---
for zip_path in zip_files:
    if not os.path.exists(zip_path):
        print(f"File not found: {zip_path}")
        continue
    
    with zipfile.ZipFile(zip_path, "r") as z:
        # Find DRUG tables
        drug_files = [f for f in z.namelist() if "DRUG" in f.upper() and f.endswith(".txt")]
        
        for file_name in drug_files:
            with z.open(file_name) as f:
                df = pd.read_csv(f, sep="$", encoding="latin-1", low_memory=False)
                
                # Drop unwanted columns if they exist
                df = df.drop(columns=[c for c in unwanted_cols if c in df.columns])
                
                # Filter only if required columns exist
                if {'drugname', 'role_cod'}.issubset(df.columns):
                    df = df[
                        (df['drugname'].str.strip().str.lower().isin(drug_list_lower)) &
                        (df['role_cod'].isin(['PS', 'SS']))
                    ]
                    
                    # Append filtered rows
                    master_drug_df = pd.concat([master_drug_df, df], ignore_index=True)


In [3]:
# --- REMOVE DUPLICATES ---
master_drug_df = master_drug_df.drop_duplicates(
    subset=['primaryid', 'caseid', 'drugname']
)


In [4]:
import pandas as pd
import numpy as np

# Convert 'Unknown' values to NaN
master_drug_df = master_drug_df.replace('Unknown', np.nan)

# Check percentage of missing values per column
missing_percent = master_drug_df.isna().mean() * 100
print("Missing value percentages per column:\n", missing_percent)

# Drop columns where more than 60% of values are missing
cols_to_drop = missing_percent[missing_percent > 60].index
print(f"Columns to drop (more than 60% missing): {list(cols_to_drop)}")

master_drug_df_clean = master_drug_df.drop(columns=cols_to_drop)

# Optional: Check the updated DataFrame
print("Remaining columns:", master_drug_df_clean.columns)


Missing value percentages per column:
 primaryid     0.000000
caseid        0.000000
drug_seq      0.000000
role_cod      0.000000
drugname      0.000000
prod_ai       0.001183
route        71.490714
dose_amt     66.206426
dose_unit    66.206426
dose_form    77.671821
dose_freq    82.844106
dtype: float64
Columns to drop (more than 60% missing): ['route', 'dose_amt', 'dose_unit', 'dose_form', 'dose_freq']
Remaining columns: Index(['primaryid', 'caseid', 'drug_seq', 'role_cod', 'drugname', 'prod_ai'], dtype='object')


In [5]:
print(f"Total rows after filtering: {len(master_drug_df)}")
master_drug_df_clean.head(100)

Total rows after filtering: 253563


,primaryid,caseid,drug_seq,role_cod,drugname,prod_ai
0,100293663,10029366,1,PS,CYCLOSPORINE,CYCLOSPORINE
2,100293663,10029366,5,SS,PREDNISOLONE,PREDNISOLONE
4,100293663,10029366,7,SS,MYCOPHENOLATE MOFETIL,MYCOPHENOLATE MOFETIL
6,101823182,10182318,2,SS,PRAMIPEXOLE,PRAMIPEXOLE\PRAMIPEXOLE DIHYDROCHLORIDE
7,103543884,10354388,2,SS,DIAZEPAM,DIAZEPAM
...,...,...,...,...,...,...
184,121237232,12123723,5,SS,AMITRIPTYLINE,AMITRIPTYLINE
185,121237402,12123740,3,SS,DIAZEPAM,DIAZEPAM
186,121237432,12123743,2,SS,DIAZEPAM,DIAZEPAM
187,121237512,12123751,5,SS,NORTRIPTYLINE,NORTRIPTYLINE


In [6]:

import pandas as pd
import zipfile

# --- Master REAC DataFrame ---
master_reac_df = pd.DataFrame()  # <- fix this line

# --- Read and clean REAC tables ---
for zip_path in zip_files:
    with zipfile.ZipFile(zip_path, "r") as z:
        reac_files = [f for f in z.namelist() if "REAC" in f.upper() and f.endswith(".txt")]

        for file_name in reac_files:
            with z.open(file_name) as f:
                reac_df = pd.read_csv(f, sep="$", encoding="latin-1", low_memory=False)

                # Remove drug_rec_act column if exists
                reac_df = reac_df.drop(columns=['drug_rec_act'], errors='ignore')

                # Keep only join-relevant columns
                reac_df = reac_df[['primaryid', 'pt']]

                master_reac_df = pd.concat([master_reac_df, reac_df], ignore_index=True)

# --- Join DRUG + REAC ---
final_df = master_drug_df_clean.merge(
    master_reac_df,
    on='primaryid',
    how='inner'
)

print("Final dataset shape:", final_df.shape)
final_df.head(10)


Final dataset shape: (2514285, 7)


,primaryid,caseid,drug_seq,role_cod,drugname,prod_ai,pt
0,100293663,10029366,1,PS,CYCLOSPORINE,CYCLOSPORINE,Toxicity to various agents
1,100293663,10029366,1,PS,CYCLOSPORINE,CYCLOSPORINE,Mycobacterium haemophilum infection
2,100293663,10029366,1,PS,CYCLOSPORINE,CYCLOSPORINE,Wound infection staphylococcal
3,100293663,10029366,1,PS,CYCLOSPORINE,CYCLOSPORINE,Staphylococcal infection
4,100293663,10029366,5,SS,PREDNISOLONE,PREDNISOLONE,Toxicity to various agents
5,100293663,10029366,5,SS,PREDNISOLONE,PREDNISOLONE,Mycobacterium haemophilum infection
6,100293663,10029366,5,SS,PREDNISOLONE,PREDNISOLONE,Wound infection staphylococcal
7,100293663,10029366,5,SS,PREDNISOLONE,PREDNISOLONE,Staphylococcal infection
8,100293663,10029366,7,SS,MYCOPHENOLATE MOFETIL,MYCOPHENOLATE MOFETIL,Toxicity to various agents
9,100293663,10029366,7,SS,MYCOPHENOLATE MOFETIL,MYCOPHENOLATE MOFETIL,Mycobacterium haemophilum infection


In [7]:
import pandas as pd
import zipfile

# Loop through ZIP files and read DEMO table
for zip_path in zip_files:
    with zipfile.ZipFile(zip_path, "r") as z:
        # Find DEMO file
        demo_files = [f for f in z.namelist() if "DEMO" in f.upper() and f.endswith(".txt")]
        
        if demo_files:
            with z.open(demo_files[0]) as f:
                demo_df = pd.read_csv(f, sep="$", encoding="latin-1", low_memory=False)
                
                print(f"\nFirst 10 records from DEMO table in {zip_path}:")  
                break  # stop after first DEMO table



First 10 records from DEMO table in faers_ascii_2024Q1.zip:


In [8]:
# Columns to drop manually
demo_cols_to_drop = [
    'rept_cod', 'to_mfr', 'caseversion', 'i_f_code', 'event_dt',
    'mfr_dt', 'init_fda_dt', 'fda_dt', 'rept_dt', 'occp_cod'
]

# Drop manually specified columns (if they exist)
demo_df_clean = demo_df.drop(
    columns=[col for col in demo_cols_to_drop if col in demo_df.columns]
)

# Drop columns where >60% values are null
threshold = 0.6  # 60% null
demo_df_clean = demo_df_clean.dropna(axis=1, thresh=int((1 - threshold) * len(demo_df_clean)))

print("Remaining DEMO columns after cleaning:")
print(demo_df_clean.columns)

# --- Join FINAL with DEMO ---
# Use suffixes to avoid automatic _x/_y column names
final_demo_df = final_df.merge(
    demo_df_clean,
    on='primaryid',
    how='inner',
    suffixes=('_drug', '_demo')  # rename overlapping columns clearly
)

# Optional: drop duplicate columns if not needed (example: caseid_demo)
if 'caseid_demo' in final_demo_df.columns:
    final_demo_df = final_demo_df.drop(columns=['caseid_demo'])

# Optional: rename columns for clarity
if 'caseid_drug' in final_demo_df.columns:
    final_demo_df = final_demo_df.rename(columns={'caseid_drug': 'caseid'})

print("Final DRUG + REAC + DEMO dataset shape:", final_demo_df.shape)
final_demo_df.head(10)

Remaining DEMO columns after cleaning:
Index(['primaryid', 'caseid', 'mfr_num', 'mfr_sndr', 'age', 'age_cod', 'sex',
       'e_sub', 'reporter_country', 'occr_country'],
      dtype='object')
Final DRUG + REAC + DEMO dataset shape: (302569, 15)


,primaryid,caseid,drug_seq,role_cod,drugname,prod_ai,pt,mfr_num,mfr_sndr,age,age_cod,sex,e_sub,reporter_country,occr_country
0,100293663,10029366,1,PS,CYCLOSPORINE,CYCLOSPORINE,Toxicity to various agents,PHHY2014AU032943,NOVARTIS,32.0,YR,M,Y,AU,AU
1,100293663,10029366,1,PS,CYCLOSPORINE,CYCLOSPORINE,Mycobacterium haemophilum infection,PHHY2014AU032943,NOVARTIS,32.0,YR,M,Y,AU,AU
2,100293663,10029366,1,PS,CYCLOSPORINE,CYCLOSPORINE,Wound infection staphylococcal,PHHY2014AU032943,NOVARTIS,32.0,YR,M,Y,AU,AU
3,100293663,10029366,1,PS,CYCLOSPORINE,CYCLOSPORINE,Staphylococcal infection,PHHY2014AU032943,NOVARTIS,32.0,YR,M,Y,AU,AU
4,100293663,10029366,5,SS,PREDNISOLONE,PREDNISOLONE,Toxicity to various agents,PHHY2014AU032943,NOVARTIS,32.0,YR,M,Y,AU,AU
5,100293663,10029366,5,SS,PREDNISOLONE,PREDNISOLONE,Mycobacterium haemophilum infection,PHHY2014AU032943,NOVARTIS,32.0,YR,M,Y,AU,AU
6,100293663,10029366,5,SS,PREDNISOLONE,PREDNISOLONE,Wound infection staphylococcal,PHHY2014AU032943,NOVARTIS,32.0,YR,M,Y,AU,AU
7,100293663,10029366,5,SS,PREDNISOLONE,PREDNISOLONE,Staphylococcal infection,PHHY2014AU032943,NOVARTIS,32.0,YR,M,Y,AU,AU
8,100293663,10029366,7,SS,MYCOPHENOLATE MOFETIL,MYCOPHENOLATE MOFETIL,Toxicity to various agents,PHHY2014AU032943,NOVARTIS,32.0,YR,M,Y,AU,AU
9,100293663,10029366,7,SS,MYCOPHENOLATE MOFETIL,MYCOPHENOLATE MOFETIL,Mycobacterium haemophilum infection,PHHY2014AU032943,NOVARTIS,32.0,YR,M,Y,AU,AU


In [9]:
#generating drug drug pairs

In [10]:
# --- Create transaction dataset ---
transactions = (
    final_demo_df[['primaryid', 'drugname']]
    .drop_duplicates()
    .assign(present=1)
    .pivot(index='primaryid', columns='drugname', values='present')
    .fillna(0)
)


In [26]:
#sanity check
print(transactions.shape)
transactions


(24297, 80)


drugname,ACYCLOVIR,ALEMTUZUMAB,AMANTADINE,AMITRIPTYLINE,AMPHOTERICIN B,AZATHIOPRINE,BACLOFEN,BENZTROPINE,BROMOCRIPTINE,CANDESARTAN,...,TIZANIDINE,TOLCAPONE,TOPIRAMATE,TRIHEXYPHENIDYL,VALPROIC ACID,VANCOMYCIN,ZOLMITRIPTAN,ZOLPIDEM,ZONISAMIDE,ZOPICLONE
primaryid,,,,,,,,,,,,,,,,,,,,,
66139682,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
73399254,False,False,False,True,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
78375095,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
79029404,False,False,False,False,False,True,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
87646472,False,False,False,True,False,False,False,False,False,False,...,False,False,False,False,False,False,False,True,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2292280312,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2309784418,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,True
2314972411,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


In [12]:
#convert to boolian
transactions = transactions.astype(bool)


In [13]:
#Run appriori algorithem
from mlxtend.frequent_patterns import apriori, association_rules


In [14]:
#resadonalbe minimam support
frequent_itemsets = apriori(
    transactions,
    min_support=0.001,   # adjust if needed
    use_colnames=True
)


In [15]:
#inspect
frequent_itemsets.sort_values('support', ascending=False).head(10)


,support,itemsets
48,0.212825,(RITUXIMAB)
35,0.089435,(MYCOPHENOLATE MOFETIL)
43,0.087171,(PREDNISOLONE)
25,0.074701,(IBUPROFEN)
22,0.068815,(GABAPENTIN)
29,0.065481,(LEVETIRACETAM)
44,0.064946,(PREGABALIN)
12,0.059555,(CYCLOSPORINE)
28,0.041939,(LAMOTRIGINE)
13,0.041857,(DIAZEPAM)


In [16]:
#keep only drug drrug items(size=2)
frequent_itemsets['itemset_size'] = frequent_itemsets['itemsets'].apply(len)

drug_drug_itemsets = frequent_itemsets[
    frequent_itemsets['itemset_size'] == 2
]


In [17]:
#generating assosiation rules (support, confidence, lift)
rules = association_rules(
    frequent_itemsets,
    metric='confidence',
    min_threshold=0.1
)


In [20]:
# Filter only drug–drug rules and make an explicit copy
rules_ddi = rules[
    (rules['antecedents'].apply(len) == 1) &
    (rules['consequents'].apply(len) == 1)
].copy()


In [21]:
#filter only drug drug rules
rules_ddi['drug_1'] = rules_ddi['antecedents'].apply(lambda x: list(x)[0])
rules_ddi['drug_2'] = rules_ddi['consequents'].apply(lambda x: list(x)[0])

rules_ddi_final = rules_ddi[
    ['drug_1', 'drug_2', 'support', 'confidence', 'lift']
].sort_values(by='lift', ascending=False)


In [24]:
rules_ddi_final

,drug_1,drug_2,support,confidence,lift
1,DIHYDROERGOTAMINE,AMITRIPTYLINE,0.001029,1.000000,52.934641
37,ETHOSUXIMIDE,CLOBAZAM,0.001029,0.454545,34.620975
139,PHENOBARBITAL,PHENYTOIN,0.002840,0.301310,33.428905
140,PHENYTOIN,PHENOBARBITAL,0.002840,0.315068,33.428905
63,DOMPERIDONE,DONEPEZIL,0.001152,0.222222,32.723232
...,...,...,...,...,...
124,MYCOPHENOLATE MOFETIL,RITUXIMAB,0.019426,0.217211,1.020611
147,PREDNISOLONE,RITUXIMAB,0.018562,0.212937,1.000527
154,ZOPICLONE,RITUXIMAB,0.004692,0.182400,0.857044
153,TOPIRAMATE,RITUXIMAB,0.004157,0.133952,0.629402


In [22]:
#apply validation threshold
validated_ddi = rules_ddi_final[
    (rules_ddi_final['support'] >= 0.001) &
    (rules_ddi_final['confidence'] >= 0.3) &
    (rules_ddi_final['lift'] > 1)
]

In [23]:
validated_ddi.head(10)
print("Number of validated DDI rules:", len(validated_ddi))


Number of validated DDI rules: 25


In [25]:
validated_ddi

,drug_1,drug_2,support,confidence,lift
1,DIHYDROERGOTAMINE,AMITRIPTYLINE,0.001029,1.000000,52.934641
37,ETHOSUXIMIDE,CLOBAZAM,0.001029,0.454545,34.620975
139,PHENOBARBITAL,PHENYTOIN,0.002840,0.301310,33.428905
140,PHENYTOIN,PHENOBARBITAL,0.002840,0.315068,33.428905
19,DOMPERIDONE,CANDESARTAN,0.001811,0.349206,16.411348
136,NORTRIPTYLINE,ZOPICLONE,0.004280,0.407843,15.854984
76,ETHOSUXIMIDE,LAMOTRIGINE,0.001482,0.654545,15.606959
95,PHENYTOIN,LACOSAMIDE,0.003087,0.342466,15.323923
61,DIHYDROERGOTAMINE,GABAPENTIN,0.001029,1.000000,14.531699
62,DIHYDROERGOTAMINE,IBUPROFEN,0.001029,1.000000,13.386777


In [ ]:
# flag ADR

In [27]:
  neuro_patterns = [
    # Seizure & epilepsy
    r'seizure', r'convulsion', r'epilep', r'status epilepticus',

    # Movement disorders
    r'tremor', r'dyskinesia', r'akathisia', r'chorea',
    r'dystonia', r'parkinson', r'parkinsonism',

    # Consciousness & cognition
    r'somnolence', r'lethargy', r'coma',
    r'loss of consciousness', r'syncope',
    r'confus', r'delirium', r'amnesia',
    r'cognitive', r'memory impairment',

    # Headache & migraine
    r'headache', r'migraine', r'cephal',

    # Balance & coordination
    r'dizziness', r'vertigo', r'ataxia',
    r'gait disturbance', r'coordination',

    # Peripheral nervous system
    r'neuropathy', r'neuralgia', r'paresthesia',
    r'hypoesthesia', r'polyneuropathy',

    # Speech & vision (neuro-related)
    r'aphasia', r'dysarthria',
    r'diplopia', r'blurred vision',

    # Autonomic / CNS disorders
    r'neuroleptic malignant syndrome',
    r'central nervous system disorder'
]


In [28]:
import re

neuro_regex = re.compile('|'.join(neuro_patterns))


In [32]:
# Replace 'pt' with the actual column name if different
final_demo_df['pt_clean'] = (
    final_demo_df['pt']  # <-- make sure this exists
    .astype(str)         # convert everything to string
    .str.lower()         # lowercase for uniformity
    .str.strip()         # remove leading/trailing spaces
)



In [33]:
import re

# Compile your neuro regex patterns first (as before)
neuro_regex = re.compile('|'.join(neuro_patterns))

# Flag cases
final_demo_df['is_neuro_adr'] = final_demo_df['pt_clean'].apply(
    lambda x: bool(neuro_regex.search(x))
)


In [34]:
final_demo_df['is_neuro_adr'] = final_demo_df['pt_clean'].apply(
    lambda x: bool(neuro_regex.search(x))
)


In [35]:
final_demo_df['is_neuro_adr'].value_counts()


False    284420
True      18149
Name: is_neuro_adr, dtype: int64

In [36]:
final_demo_df.loc[
    final_demo_df['is_neuro_adr'],
    'pt'
].drop_duplicates().sample(30)


29658                             Morton's neuralgia
41595                   Cardiac autonomic neuropathy
48                             Loss of consciousness
283053                         Listeria encephalitis
130320                           Parkinsonian crisis
181593                             Clonic convulsion
188470                          Thunderclap headache
230963                      Mononeuropathy multiplex
154425                Progressive myoclonic epilepsy
318                                             Coma
129368                   Product packaging confusion
36958                             Petit mal epilepsy
241395            Peripheral sensorimotor neuropathy
280654                          Dissociative amnesia
203794                  Demyelinating polyneuropathy
24812                  Meningoencephalitis bacterial
262273                             Axonal neuropathy
32554                    Peripheral motor neuropathy
262198           Developmental coordination di

In [37]:
# All primaryid where at least one neuro ADR occurred
neuro_cases = set(final_demo_df.loc[final_demo_df['is_neuro_adr'], 'primaryid'])


In [38]:
# Only keep primaryid and drugname
case_drug = final_demo_df[['primaryid', 'drugname']].drop_duplicates()

# Create a pivot: rows=primaryid, columns=drugname, values=1 if drug present
case_drug_matrix = case_drug.assign(present=1).pivot(
    index='primaryid', columns='drugname', values='present'
).fillna(0).astype(int)


In [39]:
pair_counts = []

for idx, row in validated_ddi.iterrows():
    drug_a = row['drug_1']
    drug_b = row['drug_2']

    # Only keep cases where columns exist in the matrix
    if drug_a not in case_drug_matrix.columns or drug_b not in case_drug_matrix.columns:
        continue

    # Boolean masks
    a_mask = case_drug_matrix[drug_a] == 1
    b_mask = case_drug_matrix[drug_b] == 1
    neuro_mask = case_drug_matrix.index.isin(neuro_cases)

    # Counts
    both_and_neuro = (a_mask & b_mask & neuro_mask).sum()
    a_only = (a_mask & ~b_mask & neuro_mask).sum()
    b_only = (~a_mask & b_mask & neuro_mask).sum()

    pair_counts.append({
        'drug_1': drug_a,
        'drug_2': drug_b,
        'A+B_neuro_ADR': both_and_neuro,
        'A_only_neuro_ADR': a_only,
        'B_only_neuro_ADR': b_only
    })

# Create a DataFrame for inspection
ddi_neuro_counts = pd.DataFrame(pair_counts)

# Optional: see top 10
ddi_neuro_counts.sort_values('A+B_neuro_ADR', ascending=False).head(10)


,drug_1,drug_2,A+B_neuro_ADR,A_only_neuro_ADR,B_only_neuro_ADR
17,LACOSAMIDE,LEVETIRACETAM,90,103,338
5,NORTRIPTYLINE,ZOPICLONE,59,23,203
12,NORTRIPTYLINE,GABAPENTIN,56,26,450
13,HALOPERIDOL,RISPERIDONE,54,86,353
24,NORTRIPTYLINE,RITUXIMAB,53,29,1177
14,PHENYTOIN,LEVETIRACETAM,43,35,385
20,VALPROIC ACID,LEVETIRACETAM,43,82,385
11,PHENOBARBITAL,LEVETIRACETAM,42,27,386
22,DOMPERIDONE,PREGABALIN,39,45,518
4,DOMPERIDONE,CANDESARTAN,38,46,178


In [40]:
# Ensure drugname is clean
final_demo_df['drugname_clean'] = final_demo_df['drugname'].str.strip().str.upper()

# Function to compute exposure totals
def compute_exposure_totals(df, drug_a, drug_b):
    # Case-level drug lists
    case_drugs = df.groupby('primaryid')['drugname_clean'].apply(set)

    A_and_B = case_drugs.apply(lambda x: drug_a in x and drug_b in x)
    A_only  = case_drugs.apply(lambda x: drug_a in x and drug_b not in x)
    B_only  = case_drugs.apply(lambda x: drug_b in x and drug_a not in x)

    return (
        A_and_B.sum(),
        A_only.sum(),
        B_only.sum()
    )


In [43]:
results = []

for _, row in ddi_neuro_counts.iterrows():
    drug_a = row['drug_1'].upper()
    drug_b = row['drug_2'].upper()

    A_B_total, A_only_total, B_only_total = compute_exposure_totals(
        final_demo_df, drug_a, drug_b
    )

    results.append({
        'drug_1': row['drug_1'],
        'drug_2': row['drug_2'],

        # Neuro ADR counts (already computed)
        'A+B_neuro_ADR': row['A+B_neuro_ADR'],
        'A_only_neuro_ADR': row['A_only_neuro_ADR'],
        'B_only_neuro_ADR': row['B_only_neuro_ADR'],

        # Exposure totals
        'A+B_total': A_B_total,
        'A_only_total': A_only_total,
        'B_only_total': B_only_total,

        # Neuro ADR rates
        'A+B_neuro_rate': row['A+B_neuro_ADR'] / A_B_total if A_B_total > 0 else 0,
        'A_only_neuro_rate': row['A_only_neuro_ADR'] / A_only_total if A_only_total > 0 else 0,
        'B_only_neuro_rate': row['B_only_neuro_ADR'] / B_only_total if B_only_total > 0 else 0
    })

ddi_rates_df = pd.DataFrame(results)


In [44]:
ddi_rates_df.sort_values(
    by='A+B_neuro_rate',
    ascending=False
).head(10)


,drug_1,drug_2,A+B_neuro_ADR,A_only_neuro_ADR,B_only_neuro_ADR,A+B_total,A_only_total,B_only_total,A+B_neuro_rate,A_only_neuro_rate,B_only_neuro_rate
22,DOMPERIDONE,PREGABALIN,39,45,518,39,87,1539,1.000000,0.517241,0.336582
4,DOMPERIDONE,CANDESARTAN,38,46,178,44,82,473,0.863636,0.560976,0.376321
21,DONEPEZIL,PREGABALIN,37,42,520,56,109,1522,0.660714,0.385321,0.341656
13,HALOPERIDOL,RISPERIDONE,54,86,353,93,207,813,0.580645,0.415459,0.434194
5,NORTRIPTYLINE,ZOPICLONE,59,23,203,104,151,521,0.567308,0.152318,0.389635
24,NORTRIPTYLINE,RITUXIMAB,53,29,1177,99,156,5072,0.535354,0.185897,0.232058
16,ZONISAMIDE,LAMOTRIGINE,21,31,283,49,114,970,0.428571,0.271930,0.291753
23,OXCARBAZEPINE,LEVETIRACETAM,28,81,400,68,156,1523,0.411765,0.519231,0.262640
14,PHENYTOIN,LEVETIRACETAM,43,35,385,111,108,1480,0.387387,0.324074,0.260135
7,PHENYTOIN,LACOSAMIDE,28,50,165,75,144,468,0.373333,0.347222,0.352564


In [45]:
len(ddi_rates_df)


25

C:\Users\User\Untitled Folder 5\faers_ascii_2024Q1\DRUG24Q1.txt
